# 📦 Phase 1 — Data Exploration & Preprocessing
## Class-Incremental Multi-Organ Segmentation (BTCV Dataset)

---

### ✅ Yêu cầu trước khi chạy notebook
Upload file **`RawData.zip`** (tải từ Synapse) lên Google Drive tại:
```
MyDrive/MultiOrganSeg/RawData.zip
```

### 🗺️ Notebook này làm gì?
| Bước | Nội dung | Output |
|------|----------|--------|
| **Step 1** | Cài thư viện & Import | Môi trường sẵn sàng |
| **Step 2** | Mount Drive + Setup thư mục | Cấu trúc thư mục |
| **Step 3** | Giải nén `RawData.zip` | `data/raw/` chứa `.nii.gz` |
| **Step 4** | 🔍 Visualize dữ liệu (CT + mask overlay) | 4 loại biểu đồ |
| **Step 5** | ⚙️ Preprocessing 3D → 2D `.npy` | `data/processed/` |
| **Step 6** | ✅ Kiểm tra bằng cách load lại file | Xác nhận OK |

---
> **💡 Cấu trúc BTCV (Synapse):**
> ```
> RawData/
> ├── Training/
> │   ├── img/    ← 30 CT có label  (img0001…img0040)
> │   └── label/  ← 30 masks        (label0001…label0040)
> └── Testing/
>     └── img/    ← 20 CT KHÔNG có label (img0061…img0080)
> ```
> Notebook này chỉ dùng **30 Training cases** (có đủ ảnh + mask).

## ⚙️ Step 1 — Cài đặt Thư viện & Import

In [ ]:
!pip install nibabel scipy tqdm -q
print("✅ Cài đặt xong!")

In [ ]:
import os, json, glob, random, time
from collections import Counter
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from scipy.ndimage import zoom
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Import thành công!")
print(f"   nibabel {nib.__version__} | numpy {np.__version__}")

## ☁️ Step 2 — Mount Google Drive & Setup Thư mục

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive đã mount tại /content/drive/")

In [ ]:
PROJECT_NAME   = "MultiOrganSeg"
DRIVE_ROOT     = f"/content/drive/MyDrive/{PROJECT_NAME}"

ZIP_PATH       = f"{DRIVE_ROOT}/RawData.zip"

RAW_DATA_DIR   = f"{DRIVE_ROOT}/data/raw"

PROCESSED_DIR  = f"{DRIVE_ROOT}/data/processed"
IMAGES_2D_DIR  = f"{PROCESSED_DIR}/images"
MASKS_2D_DIR   = f"{PROCESSED_DIR}/masks"

CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
LOG_DIR        = f"{DRIVE_ROOT}/logs"

for path in [RAW_DATA_DIR, IMAGES_2D_DIR, MASKS_2D_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(path, exist_ok=True)

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
    print(f"✅ Tìm thấy: {ZIP_PATH}  ({size_gb:.2f} GB)")
else:
    print(f"❌ Không tìm thấy: {ZIP_PATH}")
    print("   Hãy upload RawData.zip lên Drive đúng đường dẫn trên.")

print(f"\n📁 Cấu trúc thư mục:")
print(f"   Raw data  → {RAW_DATA_DIR}")
print(f"   2D images → {IMAGES_2D_DIR}")
print(f"   2D masks  → {MASKS_2D_DIR}")

## 📦 Step 3 — Giải nén `RawData.zip`

In [ ]:
# ============================================================
# Giải nén RawData.zip vào thư mục data/raw
# ⏱️  Ước tính: 5-15 phút (tùy tốc độ Drive)
# Nếu đã giải nén rồi → BỎ QUA cell này, chạy cell verify bên dưới
# ============================================================
import zipfile

print(f"📦 Bắt đầu giải nén: {os.path.basename(ZIP_PATH)}")
print(f"   Đích: {RAW_DATA_DIR}")

t0 = time.time()
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    total_files = len(zf.namelist())
    print(f"   Tổng số file trong zip: {total_files}")
    zf.extractall(RAW_DATA_DIR)

print(f"\n✅ Giải nén xong! ({(time.time()-t0)/60:.1f} phút)")

In [ ]:
# ============================================================
# Xác nhận dataset & ghép cặp image ↔ label
#
# VẤN ĐỀ: BTCV có 50 CT images (30 training + 20 testing)
#          nhưng chỉ có 30 labels (chỉ training mới có).
# GIẢI PHÁP: Trích số ID từ tên file rồi ghép cặp img ↔ label.
#   img0001.nii.gz  ↔  label0001.nii.gz  → ID = "0001" ✅
#   img0061.nii.gz  ↔  (không có label)  → bỏ qua     ❌
# ============================================================

def extract_id(filepath):
    """Lấy phần số trong tên file. Ví dụ: img0001.nii.gz → '0001'"""
    basename = os.path.basename(filepath).split('.')[0]  # bỏ đuôi .nii.gz
    return ''.join(filter(str.isdigit, basename))        # giữ lại chữ số

# Tìm toàn bộ file NIfTI
all_nii     = sorted(glob.glob(f"{RAW_DATA_DIR}/**/*.nii.gz", recursive=True) +
                     glob.glob(f"{RAW_DATA_DIR}/**/*.nii",    recursive=True))

all_images  = [f for f in all_nii if os.path.basename(f).lower().startswith('img')]
all_labels  = [f for f in all_nii if os.path.basename(f).lower().startswith('label')]

print(f"📊 Tìm thấy:")
print(f"   CT images (training + testing): {len(all_images)}")
print(f"   Label masks (training only)   : {len(all_labels)}")

# Xây bảng tra cứu: ID → đường dẫn label
label_by_id = {extract_id(f): f for f in all_labels}
print(f"\n   Label IDs có sẵn: {sorted(label_by_id.keys())}")

# Ghép cặp: chỉ giữ image có label tương ứng
paired = []
for img_f in sorted(all_images):
    img_id = extract_id(img_f)
    if img_id in label_by_id:
        paired.append((img_f, label_by_id[img_id]))

image_files = [p[0] for p in paired]
label_files = [p[1] for p in paired]

print(f"\n✅ Sau khi ghép cặp (chỉ Training):")
print(f"   CT images có label: {len(image_files)}  (kỳ vọng: 30)")
print(f"   Labels tương ứng  : {len(label_files)}  (kỳ vọng: 30)")

print(f"\n📋 5 cặp đầu tiên:")
for img_f, lbl_f in list(zip(image_files, label_files))[:5]:
    print(f"   {os.path.basename(img_f):25s} ↔  {os.path.basename(lbl_f)}")

# Kiểm tra cuối
assert len(image_files) > 0,                   "❌ Không tìm thấy file CT!"
assert len(image_files) == len(label_files),   "❌ Số cặp image-label không khớp!"
print(f"\n✅ Dataset sẵn sàng — Sẵn sàng cho Step 4!")

## 🔍 Step 4 — Visualize Dữ liệu

4 loại visualization:
1. **Overview** — 9 axial slice trải đều với mask overlay
2. **Detail** — 1 slice đẹp nhất: CT gốc | Mask thuần | Overlay + Legend
3. **Class Imbalance** — Phân bố pixel giữa 13 cơ quan
4. **3 Views** — Axial | Coronal | Sagittal

In [ ]:
ORGAN_NAMES = {
    0:  "Background",
    1:  "Spleen (Lách)",
    2:  "Right Kidney (Thận phải)",
    3:  "Left Kidney (Thận trái)",
    4:  "Gallbladder (Túi mật)",
    5:  "Esophagus (Thực quản)",
    6:  "Liver (Gan)",
    7:  "Stomach (Dạ dày)",
    8:  "Aorta (Động mạch chủ)",
    9:  "IVC (Tĩnh mạch chủ dưới)",
    10: "Portal Vein (TM cửa & lách)",
    11: "Pancreas (Tụy)",
    12: "Right Adrenal (TTT phải)",
    13: "Left Adrenal (TTT trái)",
}

ORGAN_COLORS = [
    [0.0, 0.0, 0.0, 0.0],   #  0: Background
    [0.9, 0.1, 0.1, 0.8],   #  1: Spleen
    [0.1, 0.5, 0.9, 0.8],   #  2: Right Kidney
    [0.1, 0.8, 0.9, 0.8],   #  3: Left Kidney
    [0.9, 0.8, 0.1, 0.8],   #  4: Gallbladder
    [0.9, 0.4, 0.1, 0.8],   #  5: Esophagus
    [0.6, 0.1, 0.8, 0.8],   #  6: Liver
    [0.1, 0.8, 0.4, 0.8],   #  7: Stomach
    [0.9, 0.1, 0.5, 0.8],   #  8: Aorta
    [0.3, 0.9, 0.9, 0.8],   #  9: IVC
    [0.9, 0.6, 0.8, 0.8],   # 10: Portal Vein
    [0.3, 0.9, 0.3, 0.8],   # 11: Pancreas
    [0.8, 0.5, 0.2, 0.8],   # 12: Right Adrenal
    [0.5, 0.3, 0.9, 0.8],   # 13: Left Adrenal
]

ORGAN_CMAP  = ListedColormap(ORGAN_COLORS)
NUM_CLASSES = 14

print(f"✅ Đã định nghĩa {NUM_CLASSES - 1} cơ quan + background")
for idx, name in ORGAN_NAMES.items():
    print(f"   Label {idx:2d}: {name}")

In [ ]:
# Thay EXPLORE_IDX (0-29) để xem case khác
EXPLORE_IDX = 0

img_path   = image_files[EXPLORE_IDX]
label_path = label_files[EXPLORE_IDX]

print(f"📂 Loading: {os.path.basename(img_path)}")
img_nib   = nib.load(img_path)
label_nib = nib.load(label_path)

img_vol   = img_nib.get_fdata()    # (W, H, D), float64, giá trị HU
label_vol = label_nib.get_fdata()  # (W, H, D), float64, giá trị 0-13
D = img_vol.shape[2]

print(f"\n📊 CT Volume:")
print(f"   Shape (W, H, D)  : {img_vol.shape}")
print(f"   HU range         : [{img_vol.min():.0f}, {img_vol.max():.0f}]")
print(f"   Voxel spacing(mm): {tuple(round(x,2) for x in img_nib.header.get_zooms())}")
print(f"   Số axial slices  : {D}")
print(f"\n📊 Label Volume:")
print(f"   Shape            : {label_vol.shape}")
print(f"   Labels có mặt   : {np.unique(label_vol.astype(int))}")

def hu_to_display(vol, hu_min=-1000, hu_max=1000):
    """Normalize HU → [0,1] CHỈ để hiển thị, không dùng để lưu."""
    return (np.clip(vol, hu_min, hu_max) - hu_min) / (hu_max - hu_min)

img_norm = hu_to_display(img_vol)

In [ ]:
# ============================================================
# VISUALIZE 1 — 9 axial slice trải đều (CT + mask overlay)
# ============================================================
start, end    = int(D * 0.1), int(D * 0.9)
slice_indices = np.linspace(start, end, 9, dtype=int)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle(
    f'Overview — {os.path.basename(img_path)}\n'
    f'Tổng {D} axial slices | 9 slice trải đều (CT + mask overlay)',
    fontsize=13, fontweight='bold', y=1.01
)
for ax, sl_idx in zip(axes.flat, slice_indices):
    ct_sl   = img_norm[:, :, sl_idx].T
    mask_sl = label_vol[:, :, sl_idx].T
    ax.imshow(ct_sl, cmap='gray', origin='lower', aspect='equal')
    masked = np.ma.masked_where(mask_sl == 0, mask_sl)
    ax.imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13,
              alpha=0.6, origin='lower', aspect='equal')
    organs_here = [o for o in np.unique(mask_sl.astype(int)) if o > 0]
    ax.set_title(f'Slice {sl_idx}  |  {len(organs_here)} organs', fontsize=9)
    ax.axis('off')

plt.tight_layout()
save_path = f"{DRIVE_ROOT}/viz_01_overview.png"
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {save_path}")

In [ ]:
# ============================================================
# VISUALIZE 2 — Chi tiết slice tốt nhất: CT | Mask | Overlay
# ============================================================
organ_counts = [len(np.unique(label_vol[:, :, i])) for i in range(D)]
BEST_SLICE   = int(np.argmax(organ_counts))
print(f"🔍 Slice tốt nhất: #{BEST_SLICE}  ({organ_counts[BEST_SLICE] - 1} cơ quan)")

ct_sl   = img_norm[:, :, BEST_SLICE].T
mask_sl = label_vol[:, :, BEST_SLICE].T.astype(int)
masked  = np.ma.masked_where(mask_sl == 0, mask_sl)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(f'Chi tiết Slice {BEST_SLICE} — Ba cách xem', fontsize=13, fontweight='bold')

axes[0].imshow(ct_sl, cmap='gray', origin='lower')
axes[0].set_title('① CT Scan (Grayscale)\nSáng = mô đặc / Tối = không khí', fontsize=10)
axes[0].axis('off')

axes[1].imshow(np.zeros_like(ct_sl), cmap='gray', origin='lower')
axes[1].imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13, origin='lower')
axes[1].set_title('② Segmentation Mask\nMỗi màu = 1 cơ quan', fontsize=10)
axes[1].axis('off')

axes[2].imshow(ct_sl, cmap='gray', origin='lower')
axes[2].imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13, alpha=0.55, origin='lower')
axes[2].set_title('③ Overlay (CT + Mask)', fontsize=10)
axes[2].axis('off')

organs_in_slice = [o for o in np.unique(mask_sl) if o > 0]
patches = [mpatches.Patch(color=ORGAN_COLORS[o][:3], label=f"{o}: {ORGAN_NAMES[o]}")
           for o in organs_in_slice]
fig.legend(handles=patches, loc='lower center', ncol=4,
           fontsize=9, frameon=True, bbox_to_anchor=(0.5, -0.18))

plt.tight_layout()
save_path = f"{DRIVE_ROOT}/viz_02_detail_slice{BEST_SLICE}.png"
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {save_path}")

In [ ]:
# ============================================================
# VISUALIZE 3 — Class Imbalance (vấn đề cốt lõi của đề tài!)
# ============================================================
label_int = label_vol.astype(int)
counts    = {ORGAN_NAMES[i]: int(np.sum(label_int == i)) for i in range(1, 14)}

names_s  = [k.split('(')[0].strip() for k in counts.keys()]
vals_s   = list(counts.values())
pairs    = sorted(zip(names_s, vals_s), key=lambda x: x[1])
s_names, s_vals = zip(*pairs)

bar_colors = []
for n in s_names:
    for i, name in ORGAN_NAMES.items():
        if i > 0 and name.split('(')[0].strip() == n:
            bar_colors.append(ORGAN_COLORS[i][:3])
            break

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle(
    '⚠️  Class Imbalance — Lý do cần Gradient Balancing\n'
    'Cơ quan nhỏ (Túi mật, Tuyến thượng thận) bị mô hình bỏ qua khi train thông thường!',
    fontsize=12, fontweight='bold'
)

bars = ax1.barh(s_names, s_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
ax1.set_xlabel('Số pixel', fontsize=11)
ax1.set_title('① Số pixel tuyệt đối', fontsize=11)
for bar, v in zip(bars, s_vals):
    ax1.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
             f'{v:,}', va='center', fontsize=8)
ax1.grid(axis='x', alpha=0.3)

total = sum(s_vals)
pcts  = [v / total * 100 for v in s_vals]
bars2 = ax2.barh(s_names, pcts, color=bar_colors, edgecolor='white', linewidth=0.5)
ax2.set_xlabel('Phần trăm (%)', fontsize=11)
ax2.set_title('② Tỷ lệ %', fontsize=11)
for bar, pct in zip(bars2, pcts):
    ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
             f'{pct:.2f}%', va='center', fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
save_path = f"{DRIVE_ROOT}/viz_03_class_imbalance.png"
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()

ratio = max(s_vals) / min(s_vals)
print(f"⚠️  Chênh lệch max/min: {ratio:.0f}x")
print(f"   Lớn nhất: {s_names[-1]} ({s_vals[-1]:,} px)")
print(f"   Nhỏ nhất: {s_names[0]} ({s_vals[0]:,} px)")
print(f"✅ Saved: {save_path}")

In [ ]:
# ============================================================
# VISUALIZE 4 — Axial | Coronal | Sagittal
# ============================================================
cx, cy, cz = [s // 2 for s in img_vol.shape]

views = [
    ('Axial\n(Từ trên xuống ↓)\n★ DÙNG TRONG TRAINING',
     img_norm[:, :, cz].T, label_vol[:, :, cz].T),
    ('Coronal\n(Từ trước ra sau →)',
     img_norm[:, cy, :].T, label_vol[:, cy, :].T),
    ('Sagittal\n(Từ phải sang trái ←)',
     img_norm[cx, :, :].T, label_vol[cx, :, :].T),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Ba góc nhìn CT Scan (Three Orthogonal Views)', fontsize=13, fontweight='bold')

for col, (title, ct_view, mask_view) in enumerate(views):
    mask_i = mask_view.astype(int)
    masked = np.ma.masked_where(mask_i == 0, mask_i)
    axes[0, col].imshow(ct_view, cmap='gray', origin='lower', aspect='auto')
    axes[0, col].set_title(f'{title}\n(CT only)', fontsize=10)
    axes[0, col].axis('off')
    axes[1, col].imshow(ct_view, cmap='gray', origin='lower', aspect='auto')
    axes[1, col].imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13,
                        alpha=0.55, origin='lower', aspect='auto')
    axes[1, col].set_title(f'{title}\n(CT + Mask)', fontsize=10)
    axes[1, col].axis('off')

patches = [mpatches.Patch(color=ORGAN_COLORS[i][:3],
                          label=f"{i}: {ORGAN_NAMES[i].split('(')[0].strip()}")
           for i in range(1, 14)]
fig.legend(handles=patches, loc='lower center', ncol=7,
           fontsize=8, frameon=True, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
save_path = f"{DRIVE_ROOT}/viz_04_three_views.png"
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"✅ Saved: {save_path}")
print("💡 Training chỉ dùng Axial view (★).")

## ⚙️ Step 5 — Preprocessing: 3D → 2D Axial Slices → Lưu `.npy`

```
Mỗi CT volume (W, H, D)  →  D slice (W, H)

Với mỗi slice:
  1. Clip HU về [-1000, 1000]
  2. Normalize → float32 [0.0, 1.0]
  3. Resize → 512×512  (bilinear cho CT, nearest cho mask)
  4. Bỏ slice có < 0.5% foreground pixel
  5. Lưu:  images/img0001_slice_0042.npy  (float32)
           masks/img0001_slice_0042.npy   (uint8, 0-13)
```

In [ ]:
CFG = {
    "hu_min":      -1000,
    "hu_max":       1000,
    "target_size": (512, 512),
    "min_fg_ratio": 0.005,
}
print("✅ Cấu hình Preprocessing:")
for k, v in CFG.items():
    print(f"   {k:20s}: {v}")

In [ ]:
def process_ct(slice_2d, cfg):
    """
    CT slice: (W, H) float64 HU  →  (target_H, target_W) float32 [0,1]
    Dùng bilinear (order=1) vì ảnh grayscale liên tục.
    """
    s = np.clip(slice_2d, cfg["hu_min"], cfg["hu_max"])
    s = (s - cfg["hu_min"]) / (cfg["hu_max"] - cfg["hu_min"])
    s = s.astype(np.float32)
    th, tw = cfg["target_size"]
    if s.shape != (th, tw):
        s = zoom(s, (th / s.shape[0], tw / s.shape[1]), order=1).astype(np.float32)
    return s


def process_mask(slice_2d, cfg):
    """
    Mask slice: (W, H) float64 [0-13]  →  (target_H, target_W) uint8 [0-13]
    Dùng nearest (order=0) để KHÔNG tạo giá trị trung gian (vd: 6.5)!
    """
    m = slice_2d.astype(np.uint8)
    th, tw = cfg["target_size"]
    if m.shape != (th, tw):
        m = zoom(m, (th / m.shape[0], tw / m.shape[1]), order=0).astype(np.uint8)
    return m


def has_enough_foreground(mask_2d, min_fg_ratio):
    return np.sum(mask_2d > 0) / mask_2d.size >= min_fg_ratio


# --- Test nhanh ---
test_ct   = process_ct(img_vol[:, :, BEST_SLICE], CFG)
test_mask = process_mask(label_vol[:, :, BEST_SLICE], CFG)
print(f"🧪 Test slice {BEST_SLICE}:")
print(f"   CT   → {test_ct.shape} {test_ct.dtype}  range [{test_ct.min():.3f}, {test_ct.max():.3f}]")
print(f"   Mask → {test_mask.shape} {test_mask.dtype} labels {np.unique(test_mask)}")
print("✅ OK!")

In [ ]:
# ============================================================
# CHẠY PREPROCESSING TOÀN BỘ DATASET
# ⏱️  Ước tính: 15-30 phút
# ============================================================
metadata      = []
total_saved   = 0
total_skipped = 0

print(f"🚀 Preprocessing {len(image_files)} volumes → {CFG['target_size']} .npy")
print("-" * 65)

for vol_idx, (img_f, lbl_f) in enumerate(zip(image_files, label_files)):
    vol_name = os.path.basename(img_f).split('.')[0]   # img0001

    img_v   = nib.load(img_f).get_fdata()
    label_v = nib.load(lbl_f).get_fdata()

    n_slices = img_v.shape[2]
    saved_v = skipped_v = 0
    organs_v = set()

    for sl_idx in range(n_slices):
        ct_raw   = img_v[:, :, sl_idx]
        mask_raw = label_v[:, :, sl_idx]

        if not has_enough_foreground(mask_raw, CFG["min_fg_ratio"]):
            skipped_v += 1
            continue

        ct_out   = process_ct(ct_raw, CFG)
        mask_out = process_mask(mask_raw, CFG)

        fname = f"{vol_name}_slice_{sl_idx:04d}.npy"
        np.save(os.path.join(IMAGES_2D_DIR, fname), ct_out)
        np.save(os.path.join(MASKS_2D_DIR,  fname), mask_out)

        organs_present = [int(o) for o in np.unique(mask_out) if o > 0]
        organs_v.update(organs_present)
        metadata.append({
            "filename":       fname,
            "volume_id":      vol_name,
            "slice_idx":      sl_idx,
            "organs_present": organs_present,
        })
        saved_v += 1

    total_saved   += saved_v
    total_skipped += skipped_v
    print(f"  [{vol_idx+1:2d}/{len(image_files)}] {vol_name:12s}  "
          f"saved={saved_v:3d}  skip={skipped_v:3d}  "
          f"organs={sorted(organs_v)}")

print("-" * 65)
kept_pct = total_saved/(total_saved+total_skipped)*100
print(f"\n✅ XONG!  saved={total_saved:,}  skipped={total_skipped:,}  ({kept_pct:.1f}% giữ lại)")

In [ ]:
metadata_path = f"{PROCESSED_DIR}/metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

vol_counts = Counter(r['volume_id'] for r in metadata)
print(f"✅ Saved: {metadata_path}  ({len(metadata)} records)")
print(f"\n📋 Số slice mỗi volume:")
for vol_id, cnt in sorted(vol_counts.items()):
    print(f"   {vol_id}: {cnt} slices")

## ✅ Step 6 — Kiểm tra bằng cách Load lại File `.npy`

In [ ]:
# ============================================================
# Load lại 1 cặp .npy ngẫu nhiên, kiểm tra shape/dtype/range
# và visualize để xác nhận pipeline đúng
# ============================================================
sample  = random.choice(metadata)
print(f"🔍 Sample: {sample['filename']}")
print(f"   Volume: {sample['volume_id']}  |  Slice: {sample['slice_idx']}")
print(f"   Organs: {[ORGAN_NAMES[o] for o in sample['organs_present']]}")

t0      = time.time()
ct_npy  = np.load(os.path.join(IMAGES_2D_DIR, sample['filename']))
msk_npy = np.load(os.path.join(MASKS_2D_DIR,  sample['filename']))
load_ms = (time.time() - t0) * 1000

print(f"\n📊 Kết quả:")
print(f"   CT   → {ct_npy.shape}  {ct_npy.dtype}  [{ct_npy.min():.4f}, {ct_npy.max():.4f}]")
print(f"   Mask → {msk_npy.shape} {msk_npy.dtype}  labels={list(np.unique(msk_npy))}")
print(f"   Load time: {load_ms:.1f} ms")

# Kiểm tra bằng assertion
assert ct_npy.dtype  == np.float32,           "CT dtype phải là float32"
assert msk_npy.dtype == np.uint8,             "Mask dtype phải là uint8"
assert ct_npy.shape  == CFG["target_size"],   f"CT shape phải là {CFG['target_size']}"
assert 0.0 <= ct_npy.min() and ct_npy.max() <= 1.0, "CT range phải [0, 1]"
assert msk_npy.max() <= 13,                   "Mask label phải <= 13"
print("\n✅ Tất cả assertions PASSED!")

# Visualize
masked = np.ma.masked_where(msk_npy == 0, msk_npy)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(f'✅ Verification: {sample["filename"]}', fontsize=12, fontweight='bold')

axes[0].imshow(ct_npy, cmap='gray', origin='upper')
axes[0].set_title(f'CT (float32, shape {ct_npy.shape})')
axes[0].axis('off')

axes[1].imshow(np.zeros_like(ct_npy), cmap='gray', origin='upper')
axes[1].imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13, origin='upper')
axes[1].set_title(f'Mask (uint8, labels={list(np.unique(msk_npy))})')
axes[1].axis('off')

axes[2].imshow(ct_npy, cmap='gray', origin='upper')
axes[2].imshow(masked, cmap=ORGAN_CMAP, vmin=0, vmax=13, alpha=0.5, origin='upper')
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Đo tốc độ load trên 20 mẫu ngẫu nhiên
samples    = random.sample(metadata, min(20, len(metadata)))
load_times = []
for rec in samples:
    t0 = time.time()
    np.load(os.path.join(IMAGES_2D_DIR, rec['filename']))
    np.load(os.path.join(MASKS_2D_DIR,  rec['filename']))
    load_times.append((time.time() - t0) * 1000)

avg_ms = np.mean(load_times)
print(f"⏱️  Tốc độ load .npy (20 mẫu):")
print(f"   Trung bình: {avg_ms:.1f} ms | Min: {np.min(load_times):.1f} ms | Max: {np.max(load_times):.1f} ms")

if avg_ms < 100:
    print("✅ Tốc độ tốt — DataLoader sẽ không làm chậm training.")
else:
    print("⚠️  Load hơi chậm. Cân nhắc copy processed/ vào /content trước khi train.")

In [ ]:
n_ct   = len(os.listdir(IMAGES_2D_DIR))
n_mask = len(os.listdir(MASKS_2D_DIR))

print("=" * 60)
print(" 📦 NOTEBOOK 01 — HOÀN TẤT")
print("=" * 60)
print(f"\n✅ Dữ liệu đã xử lý ({n_ct:,} slices):")
print(f"   CT slices  : {IMAGES_2D_DIR}")
print(f"   Mask slices: {MASKS_2D_DIR}")
print(f"   Metadata   : {PROCESSED_DIR}/metadata.json")
print(f"\n📊 Thông số: {CFG['target_size']} | CT=float32[0,1] | Mask=uint8[0-13]")
print(f"\n🚀 Notebook 02: Dataset → Model → Loss → Training → Dice Score")
print("=" * 60)